In [1]:
import sys, os
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import RAW_DATA, INTERIM_DATA

## Getting Data From WRDS

In [2]:
# WRDS data base
from dotenv import load_dotenv
load_dotenv()

import wrds
db = wrds.Connection(wrds_username=os.getenv('WRDS_USERNAME'), wrds_password=os.getenv('WRDS_PASSWORD'))

Loading library list...
Done


In [3]:
# Getting risk-free rate from Fama-French 3 factors
rf = db.get_table(library='ff', table='factors_daily', columns=['date', 'rf'])

rf['date'] = pd.to_datetime(rf['date'])

rf.to_parquet(RAW_DATA / 'rf.parquet')

# Daily levels of Cboe's volatility index (VIX)
vix = db.get_table(library='cboe_all', table= 'cboe', columns=['date', 'vixo', 'vixh', 'vixl', 'vix'])

vix['date'] = pd.to_datetime(vix['date'])

vix.to_parquet(RAW_DATA / 'vix.parquet')

In [4]:
display(rf, vix)

,date,rf
0,1926-07-01,0.0001
1,1926-07-02,0.0001
2,1926-07-06,0.0001
3,1926-07-07,0.0001
4,1926-07-08,0.0001
...,...,...
26166,2026-01-26,0.0002
26167,2026-01-27,0.0002
26168,2026-01-28,0.0002
26169,2026-01-29,0.0002


,date,vixo,vixh,vixl,vix
0,1986-01-02,<NA>,<NA>,<NA>,<NA>
1,1986-01-03,<NA>,<NA>,<NA>,<NA>
2,1986-01-06,<NA>,<NA>,<NA>,<NA>
3,1986-01-07,<NA>,<NA>,<NA>,<NA>
4,1986-01-08,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...
10651,2026-02-23,20.49,22.04,19.5,21.01
10652,2026-02-24,21.24,22.08,19.23,19.55
10653,2026-02-25,19.59,19.62,17.86,17.93
10654,2026-02-26,18.07,20.54,17.5,18.63


## Getting Data From yfinance

In [5]:
# getting (SPY) index daily return data
import yfinance

spy = yfinance.download('SPY', period='max', interval='1d', auto_adjust=False).reset_index()

spy.columns = spy.columns.droplevel(1).str.lower()
spy.columns.name = None

spy['date'] = pd.to_datetime(spy['date'])
spy = spy.rename(columns={
    'adj close':'spy_close_adj',
    'close':'spy_close',
    'high':'spy_high',
    'low':'spy_low',
    'open':'spy_open',
    'volume':'spy_volume'
})

spy.to_parquet(RAW_DATA / 'spy.parquet')

display(spy)

[*********************100%***********************]  1 of 1 completed


,date,spy_close_adj,spy_close,spy_high,spy_low,spy_open,spy_volume
0,1993-01-29,24.175386,43.937500,43.968750,43.750000,43.968750,1003200
1,1993-02-01,24.347321,44.250000,44.250000,43.968750,43.968750,480500
2,1993-02-02,24.398920,44.343750,44.375000,44.125000,44.218750,201300
3,1993-02-03,24.656834,44.812500,44.843750,44.375000,44.406250,529400
4,1993-02-04,24.759987,45.000000,45.093750,44.468750,44.968750,531500
...,...,...,...,...,...,...,...
8342,2026-03-23,655.380005,655.380005,662.619995,653.940002,658.070007,134802700
8343,2026-03-24,653.179993,653.179993,657.030029,649.880005,651.320007,96457500
8344,2026-03-25,656.820007,656.820007,660.890015,654.239990,658.669983,90653800
8345,2026-03-26,645.090027,645.090027,654.849976,644.820007,652.059998,96494400


# Merging data

In [6]:
spy_ = spy.sort_values('date').copy()
vix_ = vix.sort_values('date').copy()
rf_  = rf.sort_values('date').copy()

In [7]:
spy_ = spy_.dropna()
vix_ = vix_.dropna()
rf_ = rf_.dropna()

df = (
    spy
    .merge(rf, on='date', how='inner')
    .merge(vix, on='date', how='inner')
)

df

,date,spy_close_adj,spy_close,spy_high,spy_low,spy_open,spy_volume,rf,vixo,vixh,vixl,vix
0,1993-01-29,24.175386,43.937500,43.968750,43.750000,43.968750,1003200,0.0001,12.49,13.16,12.42,12.42
1,1993-02-01,24.347321,44.250000,44.250000,43.968750,43.968750,480500,0.0001,12.51,12.92,12.18,12.33
2,1993-02-02,24.398920,44.343750,44.375000,44.125000,44.218750,201300,0.0001,12.47,12.89,12.22,12.25
3,1993-02-03,24.656834,44.812500,44.843750,44.375000,44.406250,529400,0.0001,11.98,12.34,11.79,12.12
4,1993-02-04,24.759987,45.000000,45.093750,44.468750,44.968750,531500,0.0001,11.86,12.84,11.69,12.29
...,...,...,...,...,...,...,...,...,...,...,...,...
8305,2026-01-26,690.843262,692.729980,694.130005,689.919983,690.489990,60473800,0.0002,16.9,17.39,15.8,16.15
8306,2026-01-27,693.595764,695.489990,696.530029,693.570007,694.179993,55506100,0.0002,16.02,16.37,15.74,16.35
8307,2026-01-28,693.525940,695.419983,697.840027,693.940002,697.049988,61172200,0.0002,16.09,17.18,16.05,16.35
8308,2026-01-29,692.149719,694.039978,697.059998,684.830017,696.390015,97486200,0.0002,16.04,19.74,16.02,16.88


# Add return column

In [9]:
df['spy_ret_adj'] = df['spy_close_adj'].pct_change()
df['spy_ret'] = df['spy_close'].pct_change()
df

,date,spy_close_adj,spy_close,spy_high,spy_low,spy_open,spy_volume,rf,vixo,vixh,vixl,vix,spy_ret_adj,spy_ret
0,1993-01-29,24.175386,43.937500,43.968750,43.750000,43.968750,1003200,0.0001,12.49,13.16,12.42,12.42,NaN,NaN
1,1993-02-01,24.347321,44.250000,44.250000,43.968750,43.968750,480500,0.0001,12.51,12.92,12.18,12.33,0.007112,0.007112
2,1993-02-02,24.398920,44.343750,44.375000,44.125000,44.218750,201300,0.0001,12.47,12.89,12.22,12.25,0.002119,0.002119
3,1993-02-03,24.656834,44.812500,44.843750,44.375000,44.406250,529400,0.0001,11.98,12.34,11.79,12.12,0.010571,0.010571
4,1993-02-04,24.759987,45.000000,45.093750,44.468750,44.968750,531500,0.0001,11.86,12.84,11.69,12.29,0.004184,0.004184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8305,2026-01-26,690.843262,692.729980,694.130005,689.919983,690.489990,60473800,0.0002,16.9,17.39,15.8,16.15,0.005078,0.005078
8306,2026-01-27,693.595764,695.489990,696.530029,693.570007,694.179993,55506100,0.0002,16.02,16.37,15.74,16.35,0.003984,0.003984
8307,2026-01-28,693.525940,695.419983,697.840027,693.940002,697.049988,61172200,0.0002,16.09,17.18,16.05,16.35,-0.000101,-0.000101
8308,2026-01-29,692.149719,694.039978,697.059998,684.830017,696.390015,97486200,0.0002,16.04,19.74,16.02,16.88,-0.001984,-0.001984


In [10]:
# flaot 64 to float 32
float64_cols = df.select_dtypes(include=['float64']).columns
df[float64_cols] = df[float64_cols].astype('float32')

df.to_parquet(INTERIM_DATA / '01-data-spy-vix-rf.parquet')